# A — LTT: Average Risk (paper-faithful core)

In [ ]:
import numpy as np, pandas as pd
from utils.csvio import load_losses_csv, is_binary_array
from utils.testing import one_sided_binomial_pval, hoeffding_pval, holm_bonferroni

csv_path='data/sample_binary_losses.csv'  # change to your CSV
alpha_risk=0.2
alpha_mtp=0.05

ids, L, cols = load_losses_csv(csv_path)
rows=[]
for i, hp in enumerate(ids):
    losses=L[i]
    rhat=float(losses.mean()); n=len(losses)
    if is_binary_array(losses):
        p=one_sided_binomial_pval(int(losses.sum()), n, alpha_risk)
        src='binomial'
    else:
        p=hoeffding_pval(rhat, n, alpha_risk)
        src='hoeffding'
    rows.append({'hyperparam_id':hp,'mean_loss':rhat,'pval':p,'src':src})

df=pd.DataFrame(rows)
df['selected']=holm_bonferroni(df['pval'].values, alpha=alpha_mtp)
df.sort_values(['selected','mean_loss'], ascending=[False,True])